In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

import requests
from tqdm import tqdm


def download_image(url, output_folder, index):
    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()

        # Extract the file name from the URL
        file_name = os.path.join(output_folder, f"image_{index}.jpg")

        # Get the total file size for the progress bar
        total_size = int(response.headers.get("content-length", 0))

        # Use tqdm to display the download progress
        with (
            tqdm(
                total=total_size,
                unit="B",
                unit_scale=True,
                desc=f"Downloading image {index}",
            ) as pbar,
            open(file_name, "wb") as image_file,
        ):
            for chunk in response.iter_content(chunk_size=8192):
                image_file.write(chunk)
                pbar.update(len(chunk))

        print(f"Downloaded image {index}: {url}")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading image {index}: {url}")
        print(f"Error details: {e}")


def download_images_from_urls(url_file, output_folder, num_workers=4):
    # Create the output folder if it doesn't exist
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    # Read URLs from the file
    with open(url_file, "r") as file:
        urls = file.read().splitlines()

    # Use ThreadPoolExecutor to download images concurrently
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [
            executor.submit(download_image, url, output_folder, i + 1)
            for i, url in enumerate(urls)
        ]

        # Wait for all futures to complete
        for future in tqdm(
            as_completed(futures), total=len(futures), desc="Overall Progress"
        ):
            pass


if __name__ == "__main__":
    # Specify the path to the text file containing URLs
    url_file_path = "C:/Users/<user>/Desktop/coding/test/siph/images.txt"

    # Specify the output folder for downloaded images
    output_folder_path = "C:/Users/<user>/Desktop/coding/test/siph/"

    # Specify the number of workers (threads) for concurrent downloads
    num_workers = 4

    # Call the function to download images concurrently
    download_images_from_urls(url_file_path, output_folder_path, num_workers)